# NutriLens — Container classifier (bowl vs plate) · Colab

Trains Stage-1 of the portion pipeline (`05 §3`). **No manual labelling** — the data is
pulled from Open Images V7 (already box-labelled) and cropped automatically.

**One-time setup:** put **`container_train.py`** in your Google Drive at
`MyDrive/nutrilens/container_train.py`.

**Every run:** Runtime → Change runtime type → **T4 GPU**, then *Run all*. That's it —
this notebook mounts Drive and pulls the script itself. End-to-end ~10–15 min.

In [ ]:
# 1. GPU check (Runtime > Change runtime type > T4 GPU if this fails)
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), 'No GPU — set Runtime > T4 GPU'

In [ ]:
# 2. Deps. Colab already has torch/torchvision; we only add FiftyOne for the
#    Open Images download. (~1 min.)
!pip install -q fiftyone

In [ ]:
# 3. Mount Drive and pull the training script into this session.
#    (Put container_train.py at MyDrive/nutrilens/ first — one-time.)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/nutrilens'
import os, shutil
src = os.path.join(DRIVE_DIR, 'container_train.py')
assert os.path.exists(src), f'Upload container_train.py to {DRIVE_DIR}/ first'
shutil.copy(src, '/content/container_train.py')
print('Loaded container_train.py from Drive.')

In [ ]:
# 4. Collect the data: download Plate + Bowl from Open Images V7 and crop the boxes
#    into data/containers/{train,val}/{bowl,plate}. ~1500 crops/class (a few minutes).
!python /content/container_train.py --collect --per-class 1500 --data-root /content/data

In [ ]:
# 5. (optional) Peek at a few collected crops to sanity-check the labels.
import glob, random
from PIL import Image
import matplotlib.pyplot as plt
paths = []
for c in ('bowl', 'plate'):
    paths += random.sample(glob.glob(f'/content/data/containers/train/{c}/*.jpg'), 4)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, p in zip(axes.ravel(), paths):
    ax.imshow(Image.open(p)); ax.set_title(p.split('/')[-2]); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 6. Train (MobileNetV3-Small: warm up the head, then fine-tune). ~5-10 min on a T4.
#    Artifacts go straight to Drive so a disconnect doesn't lose them.
!python /content/container_train.py --train \
    --data-root /content/data --out-dir /content/drive/MyDrive/nutrilens/container_artifacts

In [ ]:
# 7. Export TorchScript + labels for the backend (loads exactly like the food model).
!python /content/container_train.py --export --out-dir /content/drive/MyDrive/nutrilens/container_artifacts
!cat /content/drive/MyDrive/nutrilens/container_artifacts/metrics.json

In [ ]:
# 8. The artifacts are already saved to your Drive at
#    MyDrive/nutrilens/container_artifacts/ . To also download them locally:
from google.colab import files
base = '{DRIVE_DIR}/container_artifacts'.replace('{DRIVE_DIR}', '/content/drive/MyDrive/nutrilens')
for f in ('container_classifier.pth', 'labels_container.json', 'metrics.json'):
    files.download(f'{base}/{f}')